<a href="https://colab.research.google.com/github/coopa33/SubjectivityMining/blob/main/Dan_preliminaries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
!pip install simpletransformers --no-deps
!pip install datasets tensorboardx scipy scikit-learn pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
simpletransformers 0.70.8 requires seqeval, which is not installed.
simpletransformers 0.70.8 requires streamlit, which is not installed.


In [61]:
!pip install transformers==5.17.0 --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 804.6/804.6 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [52]:
import torch, torchvision
print(torch.__version__, torchvision.__version__)
from simpletransformers.classification import ClassificationModel, ClassificationArgs
import pandas as pd
import random
import numpy as np
import torch
from sklearn.utils import resample
import wandb


2.11.0+cu128 0.26.0+cu128


In [56]:
# Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
if torch.mps.is_available():
    torch.mps.manual_seed(seed)

## Data

### To-Do
- Split off validation set from training data

In [57]:
# Undersampling function
def undersample(df: pd.DataFrame):
  """Undersamples to the minority class, which for all our
  data is OFF, to balance the classes"""
  df_not = df[df['labels'] == 0]
  df_off = df[df['labels'] == 1]
  df_not_resampled = resample(df_not, replace=False, n_samples=len(df_off),
                    random_state = seed)
  df_balanced = pd.concat([df_not_resampled, df_off])
  return df_balanced


In [58]:
# Load Data
olid_train = pd.read_csv("data/olid-train.csv")
olid_test = pd.read_csv("data/olid-test.csv")
olid_train_small = pd.read_csv("data/olid-train-small.csv")
hasoc_train = pd.read_csv("data/hasoc-train.csv")




In [59]:
# Do we want to balance the classes in training data?
do_undersample = False
if do_undersample:
  olid_train = undersample(olid_train)
  olid_train_small = undersample(olid_train_small)
  hasoc_train = undersample(hasoc_train)

In [60]:
# Segment IDs, also to get data in right format for model
olid_train_id = olid_train.pop("id")
olid_test_id = olid_test.pop("id")
olid_train_small_id = olid_train_small.pop("id")
hasoc_train_id = hasoc_train.pop("id")

In [61]:
# Descriptives
## Class distribution
print(f"Train shape: {olid_train_small.shape}")
print(f"Class distribution:\n{olid_train_small['labels'].value_counts()}")
print(f"Test shape: {olid_test.shape}")
print(f"Test class distribution:\n{olid_test['labels'].value_counts()}")

Train shape: (5852, 2)
Class distribution:
labels
0    3591
1    2261
Name: count, dtype: int64
Test shape: (860, 2)
Test class distribution:
labels
0    620
1    240
Name: count, dtype: int64


## Hyperparamer Tuning

### Optional Hyperparameters
- Learning rate: float
- Training Epochs
- Batch sizes
- Weight decay
- Undersampling: Yes/No
- Class penalty: [float, float]
- Gradient clipping: float

... (to be determined)

In [ ]:
model_args = ClassificationArgs(
    overwrite_output_dir=True,
)

# Check out some default values of the model/optimizer
print("[Defaults]")
print("learning rate: ", model_args.learning_rate)
print("num_train_epochs: ", model_args.num_train_epochs)

In [55]:
sweep_config = {
    "method": "bayes",  # grid, random
    "metric": {"name": "train_loss", "goal": "minimize"},
    "parameters": {
        "num_train_epochs": {"values": [2, 3, 5]},
        "learning_rate": {"min": 5e-5, "max": 4e-4},
    },
}

sweep_id = wandb.sweep(sweep_config, project="Simple Sweep")

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Create sweep with ID: gcuti6s1
Sweep URL: https://wandb.ai/dan-yu-h97-vrije-universiteit-amsterdam/Simple%20Sweep/sweeps/gcuti6s1


In [62]:

model_args.wandb_project = "Simple Sweep"
model_args.evaluate_during_training = True
model_args.reprocess_input_data = True
model_args.overwrite_output_dir = True
model_args.train_batch_size = 16
model_args.eval_batch_size = 8


In [64]:
# Training function for Tuning
def train():
  wandb.init()
  model = ClassificationModel(
    "roberta",
    "roberta-base",
    num_labels=2,
    weight=[0.5, 1],
    args=model_args)

  # Train the model
  model.train_model(olid_train_small, eval_df=olid_test)

  # Evaluate the model
  model.eval_model(olid_test)

  # Sync wandb
  wandb.join()


In [65]:
wandb.agent(sweep_id, train)


wandb: Agent Starting Run: y8zq06hv with config:
wandb: 	learning_rate: 0.0001349375239324374
wandb: 	num_train_epochs: 2
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: dan-yu-h97 (dan-yu-h97-vrije-universiteit-amsterdam) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/simpletransformers/classification/classification_model.py:544: UserWarning: The 'eval_df' parameter has been deprecated and will be removed in a future version.

  0%|          | 0/11 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

Running Evaluation:   0%|          | 0/108 [00:00<?, ?it/s]

wandb: WARNING Artifact "run-z08htasa-confusion_matrix" already exists with the same content. No new version will be created.


Training loss,▆█▃▂▃▃▁
accuracy,▁
auprc,▁
auroc,▁
eval_loss,▁
f1_score,▁
fn,▁
fp,▁
global_step,▁▂▃▄▅▇██
lr,█▇▆▅▃▂▁
+4,...


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


## Model Training

In [ ]:
model = ClassificationModel(
    "roberta",
    "roberta-base",
    num_labels=2,
    weight=[0.5, 1],
    args=model_args)

In [49]:
# Evaluate
results, model_output, wrong_predictions = model.eval_model(olid_test)


  0%|          | 0/1 [00:00<?, ?it/s]

Running Evaluation:   0%|          | 0/9 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/simpletransformers/classification/classification_model.py:1570: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


In [50]:
# Predict
predictions, raw_outputs = model.predict(["@USER How is she hiding her ugly personality. She is the worst."])

0it [00:00, ?it/s]

Predicting:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/simpletransformers/classification/classification_model.py:2260: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


In [51]:
display(results)
display(predictions)
display(raw_outputs)
"""
Model classifies everything as NOT. Overfitting?
"""

{'mcc': np.float64(0.5711260722134734),
 'accuracy': 0.8174418604651162,
 'f1_score': 0.7833853933539648,
 'tp': np.int64(181),
 'tn': np.int64(522),
 'fp': np.int64(98),
 'fn': np.int64(59),
 'auroc': np.float64(0.8836693548387098),
 'auprc': np.float64(0.765728324088248),
 'eval_loss': 0.4416522747940487}

array([1])

array([[-1.40039062,  1.32910156]])

'\nModel classifies everything as NOT. Overfitting?\n'